# Test Balldontlie API - Starting Pitchers
This notebook explores different methods to get starting pitcher information from Balldontlie API

In [3]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import json

# Load API key
load_dotenv()
API_KEY = os.getenv('BALLDONTLIE_API_KEY')

BASE_URL = "https://api.balldontlie.io/mlb/v1"
headers = {"Authorization": API_KEY}

def api_request(endpoint, params=None):
    """Make request to Balldontlie API"""
    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, headers=headers, params=params, timeout=15)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text[:200]}")
        return None

print("✅ API setup complete")

✅ API setup complete


## Method 1: Check Games Endpoint for Probable Pitchers

In [4]:
# Get games from a date we know has data
test_date = "2025-07-15"
games = api_request("/games", params={"dates[]": test_date, "per_page": 3})

if games and 'data' in games and len(games['data']) > 0:
    print(f"Found {len(games['data'])} games on {test_date}\n")
    
    # Examine first game structure
    sample_game = games['data'][0]
    print("Game structure:")
    print(json.dumps(sample_game, indent=2)[:2000])
    
    # Check if there are any pitcher-related fields
    print("\n" + "="*60)
    print("Checking for pitcher-related fields...")
    for key in sample_game.keys():
        if 'pitch' in key.lower() or 'starter' in key.lower() or 'probable' in key.lower():
            print(f"  ✅ Found: {key} = {sample_game[key]}")
else:
    print("❌ No games found")

❌ No games found


## Method 2: Check Individual Game Detail

In [ ]:
# Get detailed info for a specific game
if games and 'data' in games and len(games['data']) > 0:
    game_id = games['data'][0]['id']
    print(f"Getting details for game ID: {game_id}")
    
    game_detail = api_request(f"/games/{game_id}")
    
    if game_detail:
        print("\nGame detail structure:")
        print(json.dumps(game_detail, indent=2)[:2000])
        
        # Look for pitcher info
        print("\n" + "="*60)
        print("Searching for pitcher information...")
        found_pitcher_info = False
        
        def search_dict(d, prefix=""):
            global found_pitcher_info
            if isinstance(d, dict):
                for k, v in d.items():
                    if any(x in str(k).lower() for x in ['pitch', 'starter', 'probable', 'sp']):
                        print(f"  ✅ {prefix}{k}: {v}")
                        found_pitcher_info = True
                    if isinstance(v, (dict, list)):
                        search_dict(v, prefix + k + ".")
            elif isinstance(d, list):
                for i, item in enumerate(d[:3]):  # Check first 3 items
                    search_dict(item, prefix + f"[{i}].")
        
        search_dict(game_detail)
        
        if not found_pitcher_info:
            print("  ❌ No pitcher information found in game detail")

## Method 3: Check Plays/Plate Appearances

In [ ]:
# Check if plays or plate appearances can identify starting pitchers
if games and 'data' in games and len(games['data']) > 0:
    game_id = games['data'][0]['id']
    print(f"Checking plays for game {game_id}...\n")
    
    # Get plate appearances (first few batters)
    plate_apps = api_request("/plate_appearances", params={"game_id": game_id, "per_page": 5})
    
    if plate_apps and 'data' in plate_apps and len(plate_apps['data']) > 0:
        print("First few plate appearances:")
        for i, pa in enumerate(plate_apps['data'][:3]):
            print(f"\nPlate Appearance {i+1}:")
            
            # Look for pitcher info
            if 'pitcher' in pa:
                print(f"  ✅ Pitcher: {pa['pitcher']}")
            
            # Print some key fields
            for key in ['inning', 'batting_team', 'result', 'pitcher']:
                if key in pa:
                    print(f"  {key}: {pa[key]}")
    else:
        print("❌ No plate appearances found or endpoint not accessible")

## Method 4: Get All Active Players by Position

In [ ]:
# Get all active starting pitchers
print("Fetching active players with 'Pitcher' in position...\n")

players = api_request("/players/active", params={"per_page": 10})

if players and 'data' in players:
    # Filter for starting pitchers
    pitchers = [p for p in players['data'] if 'pitcher' in p.get('position', '').lower()]
    
    print(f"Found {len(pitchers)} pitchers in first page:")
    for p in pitchers[:5]:
        print(f"  - {p['full_name']}: {p['position']} ({p['team']['abbreviation']})")
    
    # Check if we can differentiate starters from relievers
    print("\nPosition types found:")
    positions = set(p['position'] for p in pitchers)
    for pos in sorted(positions):
        count = len([p for p in pitchers if p['position'] == pos])
        print(f"  - {pos}: {count}")
else:
    print("❌ Could not fetch players")

## Method 5: Check if Stats Endpoint Has "Games Started"

In [ ]:
# Get a pitcher's season stats to see if games_started is available
if players and 'data' in players:
    # Find a pitcher
    pitcher = next((p for p in players['data'] if 'pitcher' in p.get('position', '').lower()), None)
    
    if pitcher:
        print(f"Checking stats for {pitcher['full_name']}...\n")
        
        stats = api_request("/season_stats", params={
            "player_id": pitcher['id'],
            "season": 2025
        })
        
        if stats and 'data' in stats and len(stats['data']) > 0:
            pitcher_stats = stats['data'][0]
            print("Available stats fields:")
            
            # Look for games started
            for key in sorted(pitcher_stats.keys()):
                if 'game' in key.lower() or 'start' in key.lower():
                    print(f"  ✅ {key}: {pitcher_stats[key]}")
            
            # Show all pitching stats
            print("\nAll pitching stats:")
            pitching_keys = [k for k in pitcher_stats.keys() if k.startswith('pitching_')]
            for key in pitching_keys[:15]:
                print(f"  {key}: {pitcher_stats[key]}")

## Summary & Recommendations

In [ ]:
print("="*70)
print("STARTING PITCHER DETECTION - SUMMARY")
print("="*70)
print("\n📊 Methods Tested:")
print("  1. Games endpoint - Check for probable pitchers field")
print("  2. Game detail endpoint - Detailed game info")
print("  3. Plate appearances - Identify first pitcher of game")
print("  4. Active players by position - Get all pitchers")
print("  5. Season stats - Check for games_started field")
print("\n💡 Recommendations:")
print("  If Method 1-2 works: Use games endpoint directly")
print("  If Method 3 works: Parse plate appearances for first pitcher")
print("  If Method 5 works: Filter players by games_started > threshold")
print("  If none work: Consider hybrid approach with existing scraper")
print("="*70)